## Hypothesis 2: There are many experiments that will make the graph too dense and complex

We have stablished that there are experiments that repeat themselves. While I deleted all of the contradictory experiments, there are still repetitions. Let's see how much.


In [1]:
import pandas as pd
import dotenv
import os

from sqlalchemy import create_engine, text

from IPython.display import display

In [2]:
dotenv.load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
database = os.getenv("DB_NAME")
host = os.getenv("DB_HOST", "127.0.0.1")
port = os.getenv("DB_PORT", "3306")

connection_str = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_str)

In [3]:
pd.read_sql("SELECT * FROM experiment_score LIMIT 5", engine)

,experiment_id,score_id,score_value
0,4,1,11.61
1,4,2,12.33
2,4,3,3.33
3,4,4,5.88
4,5,1,10.94


In [4]:
pd.read_sql("SELECT * FROM score", engine)

,score_id,score_name
0,2,Bliss
1,1,HSA
2,3,Loewe
3,4,ZIP


As I concluded in the [the scores notebook](./scores.ipynb), I am going to only work with ZIP scores.


In [5]:
experiment = pd.read_sql(
    """
    SELECT 
        es.experiment_id, 
        e.dc_id,
        e.cell_line_id,
        es.score_value
    FROM experiment_score es 
    JOIN experiment e 
        ON es.experiment_id = e.experiment_id
    WHERE score_id = 4
        AND e.valid_experiment = TRUE
    """,
    engine,
)
experiment

,experiment_id,dc_id,cell_line_id,score_value
0,4,1,CVCL_1059,5.880
1,5,1,CVCL_1059,3.590
2,6,2,CVCL_1059,12.290
3,7,2,CVCL_1059,7.640
4,8,2,CVCL_1059,14.790
...,...,...,...,...
10122,10665,122,CVCL_1691,2.620
10123,10666,241,CVCL_1711,-0.400
10124,10667,238,CVCL_1557,0.650
10125,10668,158,CVCL_0459,-7.690


In [6]:
# Count the number of unique experiments per dc_id and cell_line_id
groups_info = (
    experiment.groupby(["dc_id", "cell_line_id"])
    .filter(lambda x: len(x) >= 2)
    .groupby(["dc_id", "cell_line_id"])
    .size()
    .value_counts()
    .sort_index()
    .reset_index(name="count")
    .rename(columns={"index": "size"})
)
groups_info["total_size"] = groups_info["count"] * groups_info["size"]

display(groups_info)
display(groups_info.agg({"total_size": "sum", "count": "sum"}))

,size,count,total_size
0,2,571,1142
1,3,291,873
2,4,628,2512
3,5,6,30
4,6,11,66
5,7,40,280


total_size    4903
count         1547
dtype: int64

Seen this, we can see that 4903 experiments could be reduced to 1547 experiments. This will greatly reduce the complexity of the input graph for the GNN.

The question now is, how do I reduce this? I have though of three ways of doing it (per group):

1. Average all the scores into one
2. Getting the median of the scores
3. Getting the max of the scores

The most conservative way of aggregating the experiments is by getting the median. This will ensure that we get a value that is not influenced by possible outliers. Since the experiments were not replicated a high number of times, we have no way of determining if there are outliers, and with such small data points, mean $\approx$ median.


In [7]:
final_gnn_data = experiment.groupby(["dc_id", "cell_line_id"]).agg({"score_value": "median"}).reset_index()

final_gnn_data

,dc_id,cell_line_id,score_value
0,1,CVCL_0039,-7.825
1,1,CVCL_0132,-2.220
2,1,CVCL_0201,-5.860
3,1,CVCL_0318,-6.130
4,1,CVCL_0399,-2.790
...,...,...,...
6766,254,CVCL_1628,-3.070
6767,254,CVCL_1705,-5.040
6768,254,CVCL_1711,-9.550
6769,254,CVCL_1858,-1.640


There are in fact 1547 rows on the new DF, as expected. I now have to save these results on a new table on the DB.


In [8]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gnn_experiments;"))
    conn.execute(
        text(
            """
            CREATE TABLE IF NOT EXISTS gnn_experiments (
                dc_id INT NOT NULL,
                cell_line_id CHAR(9) NOT NULL,
                score_value FLOAT NOT NULL,
                
                PRIMARY KEY (dc_id, cell_line_id),
                FOREIGN KEY (dc_id) REFERENCES drug_combination(dc_id),
                FOREIGN KEY (cell_line_id) REFERENCES cell_line(cell_line_id)
            );
            """
        )
    )

In [9]:
final_gnn_data.to_sql("gnn_experiments", con=engine, if_exists="append", index=False)

6771